# AfriCoT-VLA: `africot_full` experiment grid on GPU

This notebook runs the full controller (slow loop with the Qwen2.5-VL-7B-Instruct VLAM on a GPU, latent world model, CEM planner and safety shield) on the four scenarios and the evaluation seeds 0 to 4.

* Results and explainability logs are written to Google Drive, so the run survives a disconnect.
* The grid is resumable: a run whose result file already exists is skipped. Re-run the grid cell after a disconnect.
* Every result file records the VLAM backend (`hf`) and the model id. Mock-backend runs are never reported.

Runtime: choose a GPU runtime (Runtime > Change runtime type). The 7B model in bfloat16 needs about 17 GB of GPU memory (A100 or L4).

In [1]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/africot'
import os
os.makedirs(f'{DRIVE}/results', exist_ok=True)
os.makedirs(f'{DRIVE}/logs', exist_ok=True)
os.makedirs(f'{DRIVE}/hf_cache', exist_ok=True)

Mounted at /content/drive


Clone the repository at the shared build branch. If the repository is private, use a GitHub token in the URL or upload the repository to Drive and change `REPO`.

In [2]:
import os
from google.colab import userdata

REPO = '/content/africot-vla'
if not os.path.exists(REPO):
  try:
    tok = userdata.get('GH_TOKEN')
    !git clone -q -b claude/africot-build https://{tok}@github.com/Topmost2020/africot-vla.git {REPO}
  except userdata.SecretNotFoundError:
    print("❌ Error: 'GH_TOKEN' secret not found in Colab Secrets.")
    print("Please click the Key icon in the left sidebar, add 'GH_TOKEN' with your GitHub Token, and enable notebook access.")
    # Fallback to cloning without a token if the repository is public
    # !git clone -q -b claude/africot-build https://github.com/Topmost2020/africot-vla.git {REPO}

if os.path.exists(REPO):
  %cd {REPO}
  !git log --oneline -1


/content/africot-vla
cf3e7ed (HEAD -> claude/africot-build, origin/claude/africot-build) Document S4 and S6; release documenter lock


Install the requirements, keeping Colab's own CUDA build of PyTorch (the repository pins the CPU wheels for local runs).

In [3]:
p = 'africot/vlam.py'
s = open(p).read()
assert 'dtype="bfloat16"' in s
open(p, 'w').write(s.replace('dtype="bfloat16"', 'dtype="float16"', 1))
!grep -n 'dtype=' africot/vlam.py

97:    def __init__(self, model_id=DEFAULT_HF_MODEL, max_new_tokens=400, dtype="float16", device=None):
106:            model_id, dtype=getattr(torch, dtype), device_map=device)


In [4]:
!grep -v -E '^(torch|torchvision)==|extra-index-url' requirements.txt > /tmp/req_colab.txt
!pip install -q -r /tmp/req_colab.txt
import torch, sumo
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('SUMO_HOME', sumo.SUMO_HOME)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.4.6 which is incompatible.
torch 2.11.0+cu130 cuda True Tesla T4
SUMO_HOME /usr/local/lib/python3.13/dist-packages/sumo


In [5]:
os.environ['HF_HOME'] = f'{DRIVE}/hf_cache'          # model download cached on Drive
os.environ['AFRICOT_LOG_DIR'] = f'{DRIVE}/logs'      # explainability logs on Drive
!python -m pytest -q tests/test_smoke.py tests/test_subgoal.py

......                                                                   [100%]
6 passed in 1.83s


Run the grid. Resumable: re-run this cell after a disconnect and finished runs are skipped. `--jobs 1` because all runs share one GPU.

In [9]:
import os, shutil, glob
shutil.rmtree(f'{DRIVE}/hf_cache', ignore_errors=True)   # remove the corrupt copy
os.environ['HF_HOME'] = '/content/hf_cache'              # local disk, not Drive
os.environ['HF_HUB_DISABLE_XET'] = '1'                   # plain download path
from huggingface_hub import snapshot_download
path = snapshot_download('Qwen/Qwen2.5-VL-3B-Instruct')
from safetensors import safe_open
for f in sorted(glob.glob(path + '/*.safetensors')):
    with safe_open(f, framework='pt') as s:
        print('ok', os.path.basename(f), len(list(s.keys())), 'tensors')

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

ok model-00001-of-00002.safetensors 556 tensors
ok model-00002-of-00002.safetensors 268 tensors


In [10]:
!python run_grid.py --controllers africot_full --scenarios normal market incident outage \
    --seeds 0 1 2 3 4 --vlam-backend hf --vlam-model Qwen/Qwen2.5-VL-3B-Instruct \
    --results-dir {DRIVE}/results --jobs 1

Loading weights: 100% 824/824 [00:29<00:00, 27.64it/s]
 Retrying in 1 seconds
/content/drive/MyDrive/africot/results/africot_full__normal__0.json
Loading weights: 100% 824/824 [00:27<00:00, 29.59it/s]
 Retrying in 1 seconds
/content/drive/MyDrive/africot/results/africot_full__normal__1.json
Loading weights: 100% 824/824 [00:27<00:00, 29.64it/s]
 Retrying in 1 seconds
/content/drive/MyDrive/africot/results/africot_full__normal__2.json
Loading weights: 100% 824/824 [00:27<00:00, 30.12it/s]
 Retrying in 1 seconds
/content/drive/MyDrive/africot/results/africot_full__normal__3.json
Loading weights: 100% 824/824 [00:27<00:00, 30.30it/s]
 Retrying in 1 seconds
/content/drive/MyDrive/africot/results/africot_full__normal__4.json
Loading weights: 100% 824/824 [00:26<00:00, 30.99it/s]
 Retrying in 1 seconds
/content/drive/MyDrive/africot/results/africot_full__market__0.json
Loading weights: 100% 824/824 [00:26<00:00, 31.22it/s]
 Retrying in 1 seconds
/content/drive/MyDrive/africot/results/africot

Check what was produced: backend, model, slow-loop calls, parse failures and mean VLAM latency per run.

In [11]:
import glob, json
for f in sorted(glob.glob(f'{DRIVE}/results/africot_full__*.json')):
    d = json.load(open(f)); m = d['metrics']
    print(os.path.basename(f), d['vlam_backend'], d['config'].get('vlam_model'), 'AWT %.1f' % m['awt'],
          'calls', m.get('slow_loop_calls'), 'parse failures', m.get('vlam_parse_failures'),
          'latency %.1f s' % (d['info'].get('vlam_latency_mean_s') or float('nan')))

africot_full__incident__0.json hf Qwen/Qwen2.5-VL-3B-Instruct AWT 34.6 calls 15 parse failures 5 latency 20.5 s
africot_full__incident__1.json hf Qwen/Qwen2.5-VL-3B-Instruct AWT 29.8 calls 13 parse failures 2 latency 18.7 s
africot_full__incident__2.json hf Qwen/Qwen2.5-VL-3B-Instruct AWT 58.4 calls 13 parse failures 2 latency 18.7 s
africot_full__incident__3.json hf Qwen/Qwen2.5-VL-3B-Instruct AWT 35.3 calls 13 parse failures 0 latency 18.2 s
africot_full__incident__4.json hf Qwen/Qwen2.5-VL-3B-Instruct AWT 50.6 calls 13 parse failures 0 latency 18.1 s
africot_full__market__0.json hf Qwen/Qwen2.5-VL-3B-Instruct AWT 24.1 calls 13 parse failures 3 latency 19.3 s
africot_full__market__1.json hf Qwen/Qwen2.5-VL-3B-Instruct AWT 24.6 calls 13 parse failures 3 latency 18.7 s
africot_full__market__2.json hf Qwen/Qwen2.5-VL-3B-Instruct AWT 20.5 calls 13 parse failures 5 latency 19.7 s
africot_full__market__3.json hf Qwen/Qwen2.5-VL-3B-Instruct AWT 28.1 calls 13 parse failures 3 latency 19.6 s


Copy the result files into the repository's `results/` folder (download them, commit them on the build branch), then regenerate the tables and figures together with the baseline and ablation results:

In [12]:
!cp {DRIVE}/results/africot_full__*.json results/
!python plots.py
!cat results/summary.md

font: serif; 120 runs; wrote /content/africot-vla/results/summary.csv, /content/africot-vla/results/summary.md
  /content/africot-vla/figures/awt_by_scenario.png
  /content/africot-vla/figures/att_by_scenario.png
  /content/africot-vla/figures/aql_by_scenario.png
  /content/africot-vla/figures/throughput_by_scenario.png
  /content/africot-vla/figures/ped_wait_by_scenario.png
  /content/africot-vla/figures/world_model_training_curve.png
Mean (SD) over evaluation seeds 0-4. AWT and ATT include insertion delay (see DEVIATIONS.md).

| scenario | controller | n | AWT (s) | ATT (s) | AQL (veh) | throughput | ped wait (s) | safety violations |
|---|---|---|---|---|---|---|---|---|
| normal | fixed_time | 5 | 36.3 (10.1) | 100.8 (14.3) | 16.8 (4.8) | 1698 (26) | 19.0 (0.9) | 0 |
| normal | count_timer | 5 | 38.1 (1.3) | 104.0 (2.1) | 17.8 (0.6) | 1704 (17) | 13.0 (1.2) | 0 |
| normal | max_pressure | 5 | 17.0 (1.1) | 76.5 (1.6) | 7.6 (0.6) | 1715 (21) | 12.5 (0.8) | 0 |
| normal | africot_nowm